# Coleta do YouTube: caso Jensen Huang × NVIDIA
Período: 18/10/2026 a 18/11/2026, datas no fuso de Nova York. Vídeos em inglês (por metadado ou detecção de idioma). Uma linha por vídeo e tabelas agregadas por dia e por pregão.

In [ ]:
import os, re, json, time, hashlib, pathlib, unicodedata
import requests
import numpy as np
import pandas as pd
from dotenv import load_dotenv

def raiz_do_projeto():
    # Acha a raiz do repositório (a pasta com .git), de onde quer que o notebook rode, em qualquer computador.
    candidatos = []
    if os.environ.get("PROJETO_DIR"):                 # opcional: força o caminho pela variável de ambiente PROJETO_DIR
        candidatos.append(pathlib.Path(os.environ["PROJETO_DIR"]))
    arq = globals().get("__vsc_ipynb_file__")         # caminho do próprio notebook, quando aberto no VS Code
    if arq:
        candidatos.append(pathlib.Path(arq).parent)
    candidatos.append(pathlib.Path.cwd())             # pasta de execução do kernel
    for c in candidatos:
        try:
            c = c.resolve()
        except OSError:
            continue
        for d in [c, *c.parents]:
            if (d / ".git").exists():
                return d
    raise RuntimeError("Não achei a raiz do projeto (pasta com .git). Abra o notebook de dentro do repositório "
                       f"(pasta notebooks/) ou defina PROJETO_DIR. Tentei: {[str(c) for c in candidatos]}")


RAIZ = raiz_do_projeto()
load_dotenv(RAIZ / ".env")
CHAVES = [k.strip() for k in os.getenv("YOUTUBE_API_KEYS", "").split(",") if k.strip()]
if not CHAVES:
    raise RuntimeError("Defina YOUTUBE_API_KEYS no arquivo .env")

INICIO, FIM = "2026-10-18", "2026-11-18"   # dias no fuso de Nova York, ambos inclusos
FUSO = "America/New_York"
CONSULTA = '"Jensen Huang"'                # frase exata ("|" seria OR e "-" seria NOT na busca do YouTube)
MAX_PAGINAS = 3                            # 50 vídeos por página; 100 unidades de cota por página
TETO = 50 * MAX_PAGINAS                    # máximo de IDs por janela de busca
JANELA_SUB_HORAS = 6                       # janelas que atingem o TETO são rebuscadas em blocos desta duração (horas)
URL = "https://www.googleapis.com/youtube/v3"
FMT_UTC = "%Y-%m-%dT%H:%M:%SZ"
COTA_DIARIA_POR_CHAVE = 10_000
FECHAMENTO_BOLSA = 16                      # hora de NY em que a NYSE fecha
FERIADOS_BOLSA = []                        # feriados da NYSE (AAAA-MM-DD); nenhum entre 18/10 e 19/11/2026

RAW, PROC = RAIZ / "data" / "raw", RAIZ / "data" / "processed"
RAW.mkdir(parents=True, exist_ok=True)
PROC.mkdir(parents=True, exist_ok=True)


def salvar_json(caminho, obj):
    """Escrita atômica: se der erro no meio, o arquivo anterior continua intacto."""
    tmp = caminho.with_suffix(".tmp")
    tmp.write_text(json.dumps(obj), encoding="utf-8")
    tmp.replace(caminho)


def carregar_json(caminho, padrao):
    if not caminho.exists():
        return padrao
    try:
        return json.loads(caminho.read_text(encoding="utf-8"))
    except json.JSONDecodeError:
        caminho.replace(caminho.with_suffix(".corrompido"))
        print(f"{caminho.name} estava corrompido; guardado como .corrompido e recomeçando vazio")
        return padrao


# O nome do cache inclui um hash dos parâmetros da busca: mudou algo, o cache é outro.
PARAMS_BUSCA = {"q": CONSULTA, "type": "video", "part": "id", "maxResults": 50, "relevanceLanguage": "en"}
hash_busca = hashlib.md5(json.dumps([PARAMS_BUSCA, MAX_PAGINAS, FUSO, "janela_1d"], sort_keys=True)
                         .encode()).hexdigest()[:8]
cache_arq = RAW / f"cache_ids_{hash_busca}.json"
cache = carregar_json(cache_arq, {})

# Chave ativa lembrada entre execuções (a cota zera à meia-noite do horário do Pacífico).
estado_arq = RAW / "chave_ativa.json"
hoje_pacifico = pd.Timestamp.now(tz="America/Los_Angeles").date().isoformat()
estado = carregar_json(estado_arq, {})
indice = estado["indice"] if estado.get("dia_cota") == hoje_pacifico and estado.get("indice", 0) < len(CHAVES) else 0
print(len(CHAVES), "chave(s) carregada(s), começando pela chave", indice + 1)

4 chave(s) carregada(s), começando pela chave 1


## Funções da API
`chamar` tenta de novo em erros de rede, de servidor e de limite de taxa, e troca para a próxima chave quando a cota da atual acaba (a chave ativa é lembrada até a cota zerar). `buscar_ids` faz a busca (100 unidades de cota por página) pedindo vídeos em inglês. `subjanelas` divide uma janela em blocos menores, usada no refino dos dias que atingem o teto.


In [ ]:
MOTIVOS_COTA = {"quotaExceeded", "dailyLimitExceeded"}
MOTIVOS_TEMPORARIOS = {"backendError", "internalError", "rateLimitExceeded", "userRateLimitExceeded"}


def trocar_chave():
    """Passa para a próxima chave e a lembra. Devolve False se não há mais chaves."""
    global indice
    if indice >= len(CHAVES) - 1:
        return False
    indice += 1
    salvar_json(estado_arq, {"dia_cota": hoje_pacifico, "indice": indice})
    print(f"Cota esgotada, trocando para a chave {indice + 1}")
    return True


def chamar(endpoint, params, tentativas=5):
    falhas, espera = 0, 2
    while True:
        try:
            resp = requests.get(f"{URL}/{endpoint}", params={**params, "key": CHAVES[indice]}, timeout=30)
            r = resp.json()
        except (requests.RequestException, ValueError) as e:   # rede, timeout ou resposta que não é JSON
            falhas += 1
            if falhas >= tentativas:
                raise RuntimeError(f"{endpoint}: falha após {tentativas} tentativas ({e})") from e
            time.sleep(espera)
            espera *= 2
            continue
        if "error" not in r:
            return r
        erro = r["error"]
        motivo = (erro.get("errors") or [{}])[0].get("reason", "desconhecido")
        if motivo in MOTIVOS_COTA or "per day" in erro.get("message", ""):
            if trocar_chave():
                continue
            raise RuntimeError("Cota esgotada em todas as chaves. O cache guardou o progresso: "
                               "rode de novo depois que a cota zerar (meia-noite do Pacífico).")
        if (motivo in MOTIVOS_TEMPORARIOS or resp.status_code >= 500) and falhas + 1 < tentativas:
            falhas += 1
            time.sleep(espera)
            espera *= 2
            continue
        raise RuntimeError(f"{motivo}: {erro.get('message', '')}")


def buscar_ids(ini, fim):
    ids, token = [], None
    for _ in range(MAX_PAGINAS):
        params = {**PARAMS_BUSCA, "publishedAfter": ini, "publishedBefore": fim}
        if token:
            params["pageToken"] = token
        r = chamar("search", params)
        ids += [i["id"]["videoId"] for i in r.get("items", []) if i.get("id", {}).get("videoId")]
        token = r.get("nextPageToken")
        if not token:
            break
    return ids


def subjanelas(ini, fim):
    """Divide uma janela (strings UTC) em blocos de JANELA_SUB_HORAS horas. O último bloco termina em `fim`."""
    a, b = pd.Timestamp(ini), pd.Timestamp(fim)
    passo = pd.Timedelta(hours=JANELA_SUB_HORAS)
    cortes, t = [], a
    while t < b:
        cortes.append(t)
        t += passo
    cortes.append(b)
    return [(x.strftime(FMT_UTC), y.strftime(FMT_UTC)) for x, y in zip(cortes[:-1], cortes[1:])]


## Busca dos IDs
Uma janela por dia de Nova York (meia-noite a meia-noite, convertida para UTC), então não há margem e nenhuma janela gasta o teto de vídeos com dias vizinhos. O cache guarda cada janela já buscada, e o nome do arquivo inclui um hash dos parâmetros da busca (consulta, páginas, idioma, fuso): mudar qualquer um deles cria um cache novo em vez de reaproveitar IDs antigos. Uma execução completa custa até 32 × `MAX_PAGINAS` × 100 unidades de cota. Apague `data/raw/cache_ids_*.json` (na raiz do projeto) só se quiser refazer a busca do zero; o antigo `cache_ids.json`, de janelas de 3 dias, não é mais usado.

**Refino dos dias no teto:** os dias cuja busca atinge o teto (`TETO` = 50 × `MAX_PAGINAS` vídeos) são buscados de novo em blocos de `JANELA_SUB_HORAS` horas, num cache separado (`cache_sub_*.json`). O cache diário continua valendo, então só o refino gasta cota (até 4 blocos × `MAX_PAGINAS` × 100 unidades por dia refinado). Se algum bloco ainda atingir o teto, reduza `JANELA_SUB_HORAS` (3, por exemplo) e rode de novo.


In [ ]:
# Uma janela por dia de NY: da meia-noite local (convertida para UTC) até a meia-noite seguinte.
janelas = []
for dia in pd.date_range(INICIO, FIM):
    ini_ny = pd.Timestamp(dia.date(), tz=FUSO)
    fim_ny = pd.Timestamp((dia + pd.Timedelta(days=1)).date(), tz=FUSO)
    janelas.append((ini_ny.tz_convert("UTC").strftime(FMT_UTC), fim_ny.tz_convert("UTC").strftime(FMT_UTC)))

pendentes = [j for j in janelas if j[0] not in cache]
custo = len(pendentes) * MAX_PAGINAS * 100
disponivel = len(CHAVES) * COTA_DIARIA_POR_CHAVE
print(f"{len(janelas)} janelas, {len(pendentes)} fora do cache: até {custo} unidades de cota (disponível: {disponivel})")
if custo > disponivel:
    print("Aviso: não cabe em um dia. A busca para quando a cota acabar e continua de onde parou na próxima execução.")

ids = set()
for ini, fim in janelas:
    if ini not in cache:
        cache[ini] = buscar_ids(ini, fim)
        salvar_json(cache_arq, cache)
        time.sleep(0.2)
    ids.update(cache[ini])

# Refino: dias que atingiram o teto são buscados de novo em blocos menores (cache separado; o diário continua valendo).
sub_arq = RAW / f"cache_sub_{hash_busca}_{JANELA_SUB_HORAS}h.json"
cache_sub = carregar_json(sub_arq, {})
dias_no_teto = [(ini, fim) for ini, fim in janelas if len(cache.get(ini, [])) >= TETO]
blocos = [b for ini, fim in dias_no_teto for b in subjanelas(ini, fim)]
novos = [b for b in blocos if b[0] not in cache_sub]
print(f"{len(dias_no_teto)} dia(s) no teto de {TETO}: {len(blocos)} blocos de {JANELA_SUB_HORAS}h, "
      f"{len(novos)} fora do cache (até {len(novos) * MAX_PAGINAS * 100} unidades de cota)")
for ini, fim in blocos:
    if ini not in cache_sub:
        cache_sub[ini] = buscar_ids(ini, fim)
        salvar_json(sub_arq, cache_sub)
        time.sleep(0.2)
    ids.update(cache_sub[ini])
blocos_no_teto = [ini for ini, _ in blocos if len(cache_sub.get(ini, [])) >= TETO]
if blocos_no_teto:
    print(f"Aviso: {len(blocos_no_teto)} bloco(s) ainda no teto ({', '.join(b[:13] for b in blocos_no_teto)}). "
          f"Reduza JANELA_SUB_HORAS e rode de novo.")

ids = sorted(ids)
print(len(ids), "IDs encontrados")

32 janelas, 0 fora do cache: até 0 unidades de cota (disponível: 40000)
23 dia(s) no teto de 150: 93 blocos de 6h, 0 fora do cache (até 0 unidades de cota)
Aviso: 48 bloco(s) ainda no teto (2026-10-29T22, 2026-10-30T10, 2026-10-31T10, 2026-11-03T11, 2026-11-06T11, 2026-11-07T05, 2026-11-07T11, 2026-11-08T11, 2026-11-08T23, 2026-11-09T05, 2026-11-09T11, 2026-11-09T17, 2026-11-09T23, 2026-11-10T05, 2026-11-10T11, 2026-11-10T17, 2026-11-10T23, 2026-11-11T11, 2026-11-11T17, 2026-11-11T23, 2026-11-12T05, 2026-11-12T11, 2026-11-12T17, 2026-11-12T23, 2026-11-13T05, 2026-11-13T11, 2026-11-13T17, 2026-11-13T23, 2026-11-14T05, 2026-11-14T11, 2026-11-14T17, 2026-11-14T23, 2026-11-15T05, 2026-11-15T11, 2026-11-15T17, 2026-11-15T23, 2026-11-16T05, 2026-11-16T11, 2026-11-16T17, 2026-11-16T23, 2026-11-17T05, 2026-11-17T11, 2026-11-17T17, 2026-11-17T23, 2026-11-18T05, 2026-11-18T11, 2026-11-18T17, 2026-11-18T23). Reduza JANELA_SUB_HORAS e rode de novo.
11433 IDs encontrados


## Detalhes dos vídeos
Título, canal, data de publicação, descrição (primeiros 1.000 caracteres), tags, idioma declarado, views, curtidas e comentários. Custa 1 unidade de cota a cada 50 vídeos.

In [ ]:
linhas = []
for i in range(0, len(ids), 50):
    r = chamar("videos", {"part": "snippet,statistics", "id": ",".join(ids[i:i + 50])})
    for v in r.get("items", []):
        s, sn = v.get("statistics", {}), v["snippet"]
        linhas.append({
            "video_id": v["id"],
            "data_publicacao": sn["publishedAt"],
            "titulo": sn.get("title", ""),
            "canal": sn.get("channelTitle", ""),
            "descricao": sn.get("description", "")[:1000],
            "tags": "|".join(sn.get("tags", [])),
            "idioma_audio": sn.get("defaultAudioLanguage"),
            "idioma_padrao": sn.get("defaultLanguage"),
            "views": s.get("viewCount"),
            "curtidas": s.get("likeCount"),
            "comentarios": s.get("commentCount"),
        })
if not linhas:
    raise RuntimeError("A API não devolveu nenhum vídeo: confira a consulta, o período e as chaves.")
bruto = pd.DataFrame(linhas)
print(len(bruto), "vídeos brutos")

11433 vídeos brutos


## Limpeza, filtros e conversão de fuso
1. Remove duplicados e converte os números (curtidas ocultas e comentários desativados ficam NaN e valem 0 só no cálculo da taxa).
2. **Conversão de fuso:** `hora_utc` é a hora original da API, `hora_nova_york` é a mesma hora no fuso da bolsa, e `data` é o dia de `hora_nova_york`.
3. Corta o período de 18/10 a 18/11 usando essa `data`.
4. **Tema:** mantém o vídeo se "Jensen Huang" aparece no título, se "NVIDIA" aparece no título, ou se "Jensen Huang" aparece na descrição ou nas tags. A coluna `tema_origem` diz qual regra valeu, para auditar.
5. **Idioma:** usa `defaultAudioLanguage` ou `defaultLanguage`; se ambos estiverem vazios, detecta o idioma do título e da descrição (precisa de `pip install langdetect`). Mantém inglês e vídeos de idioma indefinido (`idioma_fonte = indefinido`); descarta o resto.
6. Calcula a taxa de engajamento e `idade_dias` (idade do vídeo na coleta, já que as views são cumulativas). O dicionário `funil` guarda quantos vídeos saíram em cada etapa. O CSV é salvo na célula seguinte.

In [ ]:
try:
    from langdetect import detect, DetectorFactory, LangDetectException
    DetectorFactory.seed = 0
    TEM_DETECTOR = True
except ImportError:
    TEM_DETECTOR = False
    print("langdetect não instalado (pip install langdetect): vídeos sem metadado de idioma ficam "
          "como 'indefinido' e são mantidos.")

PESSOA = re.compile(r"jens[eo]n\s+huang", re.I)
EMPRESA = re.compile(r"nvidia", re.I)


def como_texto(x):
    return x if isinstance(x, str) else ""


def exigir_dados(d, etapa):
    if d.empty:
        raise RuntimeError(f"Nenhum vídeo sobrou depois da etapa: {etapa}")


def origem_tema(r):
    if PESSOA.search(r["titulo"]):
        return "titulo_pessoa"
    if EMPRESA.search(r["titulo"]):
        return "titulo_empresa"
    if PESSOA.search(f"{r['descricao']} {r['tags']}"):
        return "descricao_tags"
    return None


def idioma_do_video(r):
    for c in ("idioma_audio", "idioma_padrao"):
        v = como_texto(r[c]).strip().lower()
        if v and v not in ("und", "zxx"):          # "und" e "zxx" = indeterminado / sem fala
            return v.split("-")[0], "metadata"
    if TEM_DETECTOR:
        t = re.sub(r"https?://\S+", " ", f"{r['titulo']}. {r['descricao'][:200]}")
        try:
            return detect(t), "detectado"
        except LangDetectException:
            pass
    return None, "indefinido"


funil = {"ids_encontrados": len(ids), "videos_brutos": len(bruto)}
df = bruto.drop_duplicates("video_id").copy()
funil["duplicados"] = len(bruto) - len(df)

for c in ["views", "curtidas", "comentarios"]:
    df[c] = pd.to_numeric(df[c], errors="coerce")

df["hora_utc"] = pd.to_datetime(df["data_publicacao"], utc=True)
df["hora_nova_york"] = df["hora_utc"].dt.tz_convert(FUSO)
df["data"] = df["hora_nova_york"].dt.date

no_periodo = (df["data"] >= pd.to_datetime(INICIO).date()) & (df["data"] <= pd.to_datetime(FIM).date())
funil["fora_do_periodo"] = int((~no_periodo).sum())
df = df[no_periodo].copy()
exigir_dados(df, "corte do período")

df["tema_origem"] = df.apply(origem_tema, axis=1)
funil["fora_do_tema"] = int(df["tema_origem"].isna().sum())
df = df[df["tema_origem"].notna()].copy()
exigir_dados(df, "filtro de tema")

idiomas = df.apply(idioma_do_video, axis=1, result_type="expand")
df["idioma"], df["idioma_fonte"] = idiomas[0], idiomas[1]
fora_idioma = df["idioma"].notna() & (df["idioma"] != "en")
funil["fora_do_idioma"] = int(fora_idioma.sum())
funil["idioma_indefinido_mantidos"] = int(df["idioma"].isna().sum())
df = df[~fora_idioma].copy()
exigir_dados(df, "filtro de idioma")

df["taxa_engajamento"] = (df["curtidas"].fillna(0) + df["comentarios"].fillna(0)) \
    / df["views"].where(df["views"] > 0)
df["data_coleta"] = pd.Timestamp.now(tz=FUSO).date()
df["idade_dias"] = ((pd.Timestamp.now(tz=FUSO) - df["hora_nova_york"]).dt.total_seconds() / 86400).round(1)  # idade na coleta

df = df.sort_values("hora_utc").reset_index(drop=True)
funil["videos_finais"] = len(df)
print(pd.Series(funil).to_string())

langdetect não instalado (pip install langdetect): vídeos sem metadado de idioma ficam como 'indefinido' e são mantidos.
ids_encontrados               11433
videos_brutos                 11433
duplicados                        0
fora_do_periodo                   0
fora_do_tema                   1177
fora_do_idioma                 1872
idioma_indefinido_mantidos        0
videos_finais                  8384


## Padronização de texto, features de tempo e outliers
- `dia_semana` e `hora_publicacao` vêm do horário de Nova York.
- `data_pregao` é o pregão da NYSE que "recebe" o vídeo: o próprio dia, se for dia útil e a publicação for antes das 16h; caso contrário, o próximo dia útil. Assim, vídeos de fim de semana e do fim da tarde entram no cruzamento com a bolsa em vez de serem descartados. Feriados da bolsa vão em `FERIADOS_BOLSA` (não há nenhum entre 18/10 e 19/11/2026).
- `titulo_limpo` fica em minúsculas, sem acentos e sem emojis ou pontuação. O `titulo` original é mantido.
- `outlier_views` marca vídeos com views muito acima dos demais, pela regra do intervalo interquartil aplicada ao log10 das views. **Decisão:** os outliers são marcados e mantidos, e a análise compara os resultados com e sem eles, porque um vídeo viral pode distorcer as somas e médias do dia. Atenção: os vídeos virais tendem a se concentrar nos dias do caso, então tirá-los tira justamente o efeito que se quer medir. Use a versão sem outliers (`views_sem_outlier`, `views_sem_outliers` nas tabelas) como teste de robustez, não como resultado principal. A célula imprime quantos outliers há por dia.
- Esta célula salva `data/raw/youtube_videos.csv`.
- `sobre_o_caso` marca os vídeos cujo **título** cita o escândalo (espionagem, relacionamento, investigação, renúncia, #BringJensenBack), por palavras-chave (`CASO`). A regra é simples de propósito: confira a amostra na seção Conferência e ajuste o regex se precisar. Há tabelas agregadas só com esses vídeos.


In [ ]:
bdia = pd.offsets.CustomBusinessDay(holidays=FERIADOS_BOLSA)

# Títulos que citam o escândalo. "intelligence" sozinho NÃO entra: pegaria "artificial intelligence".
CASO = re.compile(
    r"\bspy\b|\bspies\b|\baffair\b|chinese intelligence|intelligence (?:asset|operative|agent|operation|effort)|"
    r"resign|steps? down|stepping down|\bousted?\b|scandal|investigat|interim ceo|bring\s*jensen\s*back",
    re.I)


def data_pregao(ts):
    """Pregão que recebe a publicação: o mesmo dia (útil, antes do fechamento) ou o próximo dia útil."""
    d = pd.Timestamp(ts.date())
    if ts.hour >= FECHAMENTO_BOLSA or not bdia.is_on_offset(d):
        return (d + bdia).date()
    return d.date()


df["dia_semana"] = df["hora_nova_york"].dt.day_name()
df["hora_publicacao"] = df["hora_nova_york"].dt.hour
df["data_pregao"] = df["hora_nova_york"].apply(data_pregao)


def limpar_titulo(t):
    t = unicodedata.normalize("NFKD", str(t))
    t = "".join(c for c in t if unicodedata.category(c) != "Mn")
    t = re.sub(r"[^\w\s]", " ", t.lower())
    return re.sub(r"\s+", " ", t).strip()


df["titulo_limpo"] = df["titulo"].apply(limpar_titulo)
df["sobre_o_caso"] = df["titulo"].str.contains(CASO)

log_views = np.log10(df["views"] + 1)
q1, q3 = log_views.quantile([0.25, 0.75])
df["outlier_views"] = log_views > q3 + 1.5 * (q3 - q1)
df["views_sem_outlier"] = df["views"].where(~df["outlier_views"])
print(df["outlier_views"].sum(), "vídeos marcados como outlier de views")
print((df["data_pregao"] != df["data"]).sum(), "vídeos deslocados para o pregão seguinte (fim de semana ou após o fechamento)")

print(int(df["sobre_o_caso"].sum()), "vídeos marcados como sobre_o_caso")
print("Outliers de views por dia (NY):")
print(df[df["outlier_views"]].groupby("data").size().to_string())

df.to_csv(RAW / "youtube_videos.csv", index=False)

3 vídeos marcados como outlier de views
4262 vídeos deslocados para o pregão seguinte (fim de semana ou após o fechamento)
2067 vídeos marcados como sobre_o_caso
Outliers de views por dia (NY):
data
2026-10-23    1
2026-11-06    1
2026-11-13    1


## Agregação por dia e por pregão
A taxa é ponderada: soma(curtidas + comentários) / soma(views). Dias sem vídeo aparecem com zeros e taxa vazia, em vez de sumirem.
- `youtube_diario.csv`: agregado por `data` (dia de Nova York). `dia_util` marca segunda a sexta.
- `youtube_diario_pregao.csv`: agregado por `data_pregao`, para o join com a bolsa. `cobertura_completa` é falso nas pontas: o primeiro pregão (19/10) não recebe a sexta depois do fechamento nem o sábado, porque a coleta começa em 18/10, e o pregão de 19/11 só recebe o que foi publicado depois do fechamento de 18/11. Exclua ou sinalize esses dias na análise.
- Colunas novas nas tabelas: `n_outliers` e `views_sem_outliers` (teste de robustez) e `idade_media_dias` (idade média dos vídeos na coleta, vazia nos dias sem vídeo).
- `youtube_diario_caso.csv` e `youtube_diario_pregao_caso.csv`: as mesmas tabelas, só com os vídeos `sobre_o_caso`. O calendário é o mesmo das tabelas completas, então os dois joins com a bolsa funcionam igual.


In [ ]:
def agregar(base, coluna, calendario):
    g = base.groupby(coluna).agg(
        n_videos=("video_id", "count"), views=("views", "sum"),
        curtidas=("curtidas", "sum"), comentarios=("comentarios", "sum"),
        n_outliers=("outlier_views", "sum"), views_sem_outliers=("views_sem_outlier", "sum"),
        idade_media_dias=("idade_dias", "mean"))
    g.index = pd.to_datetime(g.index)
    g = g.reindex(calendario, fill_value=0)          # dia sem vídeo vira linha de zeros
    g.loc[g["n_videos"] == 0, "idade_media_dias"] = np.nan   # ...menos a idade média, que fica vazia
    g["idade_media_dias"] = g["idade_media_dias"].round(1)
    g["taxa_engajamento"] = (g["curtidas"] + g["comentarios"]) / g["views"].where(g["views"] > 0)
    g.index.name = coluna
    return g.reset_index()


# Por dia de calendário (NY)
diario = agregar(df, "data", pd.date_range(INICIO, FIM))
diario["dia_util"] = diario["data"].dt.dayofweek < 5
diario["data"] = diario["data"].dt.date
diario.to_csv(PROC / "youtube_diario.csv", index=False)

# Só os vídeos sobre o caso, no mesmo calendário
caso = df[df["sobre_o_caso"]]
diario_caso = agregar(caso, "data", pd.date_range(INICIO, FIM))
diario_caso["dia_util"] = diario_caso["data"].dt.dayofweek < 5
diario_caso["data"] = diario_caso["data"].dt.date
diario_caso.to_csv(PROC / "youtube_diario_caso.csv", index=False)

# Por pregão da NYSE
ultimo = max(pd.Timestamp(FIM), pd.Timestamp(df["data_pregao"].max()))
pregoes = pd.bdate_range(INICIO, ultimo, freq="C", holidays=FERIADOS_BOLSA)
diario_pregao = agregar(df, "data_pregao", pregoes)
anterior = pd.Series([d - bdia for d in diario_pregao["data_pregao"]])
diario_pregao["cobertura_completa"] = ((anterior >= pd.Timestamp(INICIO)).values
                                       & (diario_pregao["data_pregao"] <= pd.Timestamp(FIM)).values)
diario_pregao["data_pregao"] = diario_pregao["data_pregao"].dt.date
diario_pregao.to_csv(PROC / "youtube_diario_pregao.csv", index=False)

diario_pregao_caso = agregar(caso, "data_pregao", pregoes)
diario_pregao_caso["cobertura_completa"] = diario_pregao["cobertura_completa"].values   # mesmo calendário
diario_pregao_caso["data_pregao"] = diario_pregao_caso["data_pregao"].dt.date
diario_pregao_caso.to_csv(PROC / "youtube_diario_pregao_caso.csv", index=False)

print(len(diario), "dias |", int(diario["dia_util"].sum()), "dias úteis |",
      len(diario_pregao), "pregões,", int(diario_pregao["cobertura_completa"].sum()), "com cobertura completa")
print(len(caso), "vídeos sobre o caso em", int((diario_caso["n_videos"] > 0).sum()), "dias")


32 dias | 23 dias úteis | 24 pregões, 22 com cobertura completa
2067 vídeos sobre o caso em 21 dias


## Conferência
Confira o menor e o maior dia, os valores ausentes e uma amostra de títulos (para ver se há ruído fora do tema ou fora do inglês). Mostra também uma amostra de títulos marcados e não marcados como `sobre_o_caso`, para checar a regra.


In [ ]:
print(df["data"].min(), "->", df["data"].max())
print(df[["views", "curtidas", "comentarios"]].isna().sum())
print("sobre_o_caso:", df["sobre_o_caso"].value_counts().to_dict())
for rotulo, mascara in [("MARCADOS (sobre_o_caso = True)", df["sobre_o_caso"]),
                        ("NÃO MARCADOS (amostra)", ~df["sobre_o_caso"])]:
    sub = df[mascara]
    print(f"\n{rotulo}:")
    print(sub.sample(min(12, len(sub)), random_state=0)[["data", "titulo", "views"]].to_string(index=False, max_colwidth=70))

df[["hora_utc", "hora_nova_york", "data", "data_pregao", "titulo", "canal", "idioma", "tema_origem",
    "views", "curtidas", "comentarios", "taxa_engajamento"]].head(10)

2026-10-18 -> 2026-11-18
views            0
curtidas       162
comentarios     48
dtype: int64
sobre_o_caso: {False: 6317, True: 2067}

MARCADOS (sobre_o_caso = True):
      data                                                                 titulo  views
2026-11-09                       xQc Jensen Huang's Affair EXPOSED | JUICE UP #42  41073
2026-11-12 Nvidia’s Jensen Huang Resigns; Trump Wants Ex-Apple Boss Tim Cook a...  34374
2026-11-13   Jensen Huang Just Resigned As Nvidia CEO… Fair Or Too Far? 😳 #Shorts   1031
2026-11-18 "Jensen Huang Had NO IDEA What Hit Him" -Former Russian Sex Spy Rev...   2100
2026-11-15           Nvidia CEO Resigns: What Jensen Huang Does With $140 Billion  14440
2026-11-08                         US Memo Claims Spy Targeted Nvidia CEO #Shorts    664
2026-11-10            Everything You Need To Know About Jensen Huang's Spy Affair    404
2026-11-11 Lawmakers threaten subpoena of Jensen Huang over alleged Chinese sp...  79137
2026-11-06                     

,hora_utc,hora_nova_york,data,data_pregao,titulo,canal,idioma,tema_origem,views,curtidas,comentarios,taxa_engajamento
0,2026-10-18 04:00:19+00:00,2026-10-18 00:00:19-04:00,2026-10-18,2026-10-19,Jensen Huang on China's AI labs | Lex Fridman ...,Lex Clips,en,titulo_pessoa,27533,230.0,26.0,0.009298
1,2026-10-18 04:24:29+00:00,2026-10-18 00:24:29-04:00,2026-10-18,2026-10-19,Jensen Huang & A.I. | An Rant from a Leftist,Kiara Anais,en,titulo_pessoa,196,4.0,5.0,0.045918
2,2026-10-18 04:30:03+00:00,2026-10-18 00:30:03-04:00,2026-10-18,2026-10-19,Jensen Huang's parents regret his gift #shorts,The Build Story,en,titulo_pessoa,1478,8.0,0.0,0.005413
3,2026-10-18 04:53:26+00:00,2026-10-18 00:53:26-04:00,2026-10-18,2026-10-19,The Truth About Success Nobody Wants to Hear,YOUCAN,en,descricao_tags,181002,5537.0,76.0,0.031011
4,2026-10-18 05:03:54+00:00,2026-10-18 01:03:54-04:00,2026-10-18,2026-10-19,"A $3,500 Photoshoot for $200? #shorts",Knowledge Egg,en,descricao_tags,735,6.0,0.0,0.008163
5,2026-10-18 05:43:13+00:00,2026-10-18 01:43:13-04:00,2026-10-18,2026-10-19,Who pays back trillions in AI capex? Jensen Huang,AI & Money,en,titulo_pessoa,214,1.0,0.0,0.004673
6,2026-10-18 06:02:06+00:00,2026-10-18 02:02:06-04:00,2026-10-18,2026-10-19,Huang Praises Microsoft’s Nvidia-Powered Lapto...,IBI Laptop News,en,titulo_empresa,24,0.0,0.0,0.000000
7,2026-10-18 06:15:15+00:00,2026-10-18 02:15:15-04:00,2026-10-18,2026-10-19,How Jensen Huang Built a $3T Empire Washing Di...,Naked Founders,en,titulo_pessoa,11,0.0,0.0,0.000000
8,2026-10-18 07:10:07+00:00,2026-10-18 03:10:07-04:00,2026-10-18,2026-10-19,"Uptown Funk ft. Elon musk, Jensen Huang, Sam A...",Stealthy Li,en,titulo_pessoa,3027,43.0,9.0,0.017179
9,2026-10-18 07:11:05+00:00,2026-10-18 03:11:05-04:00,2026-10-18,2026-10-19,Will AI take my job? Ask an economist! | The I...,NPR's Planet Money,en,descricao_tags,3024,6.0,1.0,0.002315


In [ ]:
diario

,data,n_videos,views,curtidas,comentarios,n_outliers,views_sem_outliers,idade_media_dias,taxa_engajamento,dia_util
0,2026-10-18,75,2338829,43192.0,4788.0,0,2338829.0,33.1,0.020515,False
1,2026-10-19,50,669357,5718.0,478.0,0,669357.0,32.1,0.009257,True
2,2026-10-20,72,1183834,24732.0,947.0,0,1183834.0,31.1,0.021691,True
3,2026-10-21,96,5464060,45617.0,4735.0,0,5464060.0,30.0,0.009215,True
4,2026-10-22,123,7631163,45368.0,3645.0,0,7631163.0,29.1,0.006423,True
5,2026-10-23,87,22127513,51543.0,1467.0,1,2693297.0,28.2,0.002396,True
6,2026-10-24,65,5427942,44602.0,5688.0,0,5427942.0,27.1,0.009265,False
7,2026-10-25,88,4679346,78069.0,3569.0,0,4679346.0,26.1,0.017446,False
8,2026-10-26,109,6895995,122239.0,1736.0,0,6895995.0,25.1,0.017978,True
9,2026-10-27,58,312974,5438.0,240.0,0,312974.0,24.1,0.018142,True


In [ ]:
diario_pregao

,data_pregao,n_videos,views,curtidas,comentarios,n_outliers,views_sem_outliers,idade_media_dias,taxa_engajamento,cobertura_completa
0,2026-10-19,113,2988834,48776.0,5249.0,0,2988834.0,32.8,0.018076,False
1,2026-10-20,67,1192870,24769.0,947.0,0,1192870.0,31.3,0.021558,True
2,2026-10-21,82,1063867,9565.0,1465.0,0,1063867.0,30.3,0.010368,True
3,2026-10-22,119,11936939,80923.0,6895.0,0,11936939.0,29.4,0.007357,True
4,2026-10-23,109,22146078,51302.0,1421.0,1,2711862.0,28.4,0.002381,True
5,2026-10-26,248,15243736,203359.0,10395.0,0,15243736.0,26.2,0.014022,True
6,2026-10-27,68,2149185,47651.0,919.0,0,2149185.0,24.5,0.022599,True
7,2026-10-28,86,3204140,3311.0,5892.0,0,3204140.0,23.3,0.002872,True
8,2026-10-29,84,310203,3622.0,409.0,0,310203.0,22.4,0.012995,True
9,2026-10-30,269,6282778,104755.0,6668.0,0,6282778.0,21.4,0.017735,True


In [ ]:
diario_caso

,data,n_videos,views,curtidas,comentarios,n_outliers,views_sem_outliers,idade_media_dias,taxa_engajamento,dia_util
0,2026-10-18,0,0,0.0,0.0,0,0.0,NaN,NaN,False
1,2026-10-19,0,0,0.0,0.0,0,0.0,NaN,NaN,True
2,2026-10-20,0,0,0.0,0.0,0,0.0,NaN,NaN,True
3,2026-10-21,0,0,0.0,0.0,0,0.0,NaN,NaN,True
4,2026-10-22,0,0,0.0,0.0,0,0.0,NaN,NaN,True
5,2026-10-23,0,0,0.0,0.0,0,0.0,NaN,NaN,True
6,2026-10-24,0,0,0.0,0.0,0,0.0,NaN,NaN,False
7,2026-10-25,0,0,0.0,0.0,0,0.0,NaN,NaN,False
8,2026-10-26,0,0,0.0,0.0,0,0.0,NaN,NaN,True
9,2026-10-27,0,0,0.0,0.0,0,0.0,NaN,NaN,True


In [ ]:
diario_pregao_caso

,data_pregao,n_videos,views,curtidas,comentarios,n_outliers,views_sem_outliers,idade_media_dias,taxa_engajamento,cobertura_completa
0,2026-10-19,0,0,0.0,0.0,0,0.0,NaN,NaN,False
1,2026-10-20,0,0,0.0,0.0,0,0.0,NaN,NaN,True
2,2026-10-21,0,0,0.0,0.0,0,0.0,NaN,NaN,True
3,2026-10-22,0,0,0.0,0.0,0,0.0,NaN,NaN,True
4,2026-10-23,0,0,0.0,0.0,0,0.0,NaN,NaN,True
5,2026-10-26,0,0,0.0,0.0,0,0.0,NaN,NaN,True
6,2026-10-27,0,0,0.0,0.0,0,0.0,NaN,NaN,True
7,2026-10-28,0,0,0.0,0.0,0,0.0,NaN,NaN,True
8,2026-10-29,12,6145,27.0,4.0,0,6145.0,22.1,0.005045,True
9,2026-10-30,62,2792391,49689.0,3576.0,0,2792391.0,21.4,0.019075,True


In [ ]:
for ini, fim in janelas:
    n = len(cache.get(ini, []))
    if n >= TETO:
        subs = subjanelas(ini, fim)
        total = len(set().union(*[cache_sub.get(i, []) for i, _ in subs]))
        ainda = sum(len(cache_sub.get(i, [])) >= TETO for i, _ in subs)
        print(ini[:10], n, f"<- teto atingido; refinado em {len(subs)} blocos: {total} IDs, {ainda} bloco(s) ainda no teto")
    else:
        print(ini[:10], n)


2026-10-18 122
2026-10-19 61
2026-10-20 102
2026-10-21 130
2026-10-22 150 <- teto atingido; refinado em 4 blocos: 162 IDs, 0 bloco(s) ainda no teto
2026-10-23 100
2026-10-24 76
2026-10-25 136
2026-10-26 150 <- teto atingido; refinado em 4 blocos: 149 IDs, 0 bloco(s) ainda no teto
2026-10-27 69
2026-10-28 120
2026-10-29 150 <- teto atingido; refinado em 4 blocos: 240 IDs, 1 bloco(s) ainda no teto
2026-10-30 150 <- teto atingido; refinado em 4 blocos: 508 IDs, 1 bloco(s) ainda no teto
2026-10-31 150 <- teto atingido; refinado em 4 blocos: 414 IDs, 1 bloco(s) ainda no teto
2026-11-01 150 <- teto atingido; refinado em 5 blocos: 370 IDs, 0 bloco(s) ainda no teto
2026-11-02 150 <- teto atingido; refinado em 4 blocos: 372 IDs, 0 bloco(s) ainda no teto
2026-11-03 150 <- teto atingido; refinado em 4 blocos: 374 IDs, 1 bloco(s) ainda no teto
2026-11-04 150 <- teto atingido; refinado em 4 blocos: 373 IDs, 0 bloco(s) ainda no teto
2026-11-05 150 <- teto atingido; refinado em 4 blocos: 363 IDs, 0 b

## Validação dos CSVs salvos
Lê os arquivos que serão entregues e confere duplicados, valores impossíveis, somas entre as tabelas, o fuso e a coerência de `data` e `data_pregao`. Os links de três vídeos sorteados servem para conferir título, canal e views à mão no YouTube.

In [ ]:
salvo = pd.read_csv(RAW / "youtube_videos.csv")
diario_salvo = pd.read_csv(PROC / "youtube_diario.csv")
pregao_salvo = pd.read_csv(PROC / "youtube_diario_pregao.csv")
diario_caso_salvo = pd.read_csv(PROC / "youtube_diario_caso.csv")
pregao_caso_salvo = pd.read_csv(PROC / "youtube_diario_pregao_caso.csv")

print("vídeos:", len(salvo), "| dias:", len(diario_salvo), "| pregões:", len(pregao_salvo))
print("video_id duplicados:", salvo["video_id"].duplicated().sum())
print("views negativas:", (salvo["views"] < 0).sum())
print("curtidas maiores que views:", (salvo["curtidas"] > salvo["views"]).sum())
print("taxa fora de 0 a 1:", (~salvo["taxa_engajamento"].dropna().between(0, 1)).sum())
for nome, t in [("diário", diario_salvo), ("pregão", pregao_salvo)]:
    print(f"tabela {nome}: views iguais às dos vídeos:", round(salvo["views"].sum()) == round(t["views"].sum()),
          "| soma de n_videos == nº de vídeos:", len(salvo) == t["n_videos"].sum())

n_caso = int(salvo["sobre_o_caso"].sum())
for nome, t in [("diário", diario_caso_salvo), ("pregão", pregao_caso_salvo)]:
    print(f"tabela {nome} (caso): soma de n_videos == vídeos sobre_o_caso:", n_caso == t["n_videos"].sum())
for nome, t in [("diário", diario_salvo), ("pregão", pregao_salvo)]:
    print(f"tabela {nome}: n_outliers == vídeos com outlier_views:", int(salvo["outlier_views"].sum()) == t["n_outliers"].sum())
print("idade_dias negativa:", (salvo["idade_dias"] < 0).sum())

salvo["fuso"] = salvo["hora_nova_york"].str[-6:]
print(salvo.groupby("fuso")["data"].agg(["min", "max"]))      # -04:00 até 31/10 e -05:00 a partir de 01/11
ny = pd.to_datetime(salvo["hora_utc"], utc=True).dt.tz_convert(FUSO)
print("data confere com hora_utc convertida:", (ny.dt.strftime("%Y-%m-%d") == salvo["data"]).all())
print("data_pregao sempre em dia útil:", (pd.to_datetime(salvo["data_pregao"]).dt.dayofweek < 5).all())
print("data_pregao nunca antes de data:", (salvo["data_pregao"] >= salvo["data"]).all())

print(salvo["idioma"].value_counts(dropna=False).head(8))
print(salvo["tema_origem"].value_counts())
print(salvo.nlargest(5, "views")[["data", "titulo", "canal", "views"]])

for _, r in salvo.sample(min(3, len(salvo)), random_state=1).iterrows():
    print(r["titulo"][:60], "|", r["canal"], "|", r["data"], "|", r["views"], "|",
          "https://www.youtube.com/watch?v=" + r["video_id"])

vídeos: 8384 | dias: 32 | pregões: 24
video_id duplicados: 0
views negativas: 0
curtidas maiores que views: 3
taxa fora de 0 a 1: 8
tabela diário: views iguais às dos vídeos: True | soma de n_videos == nº de vídeos: True
tabela pregão: views iguais às dos vídeos: True | soma de n_videos == nº de vídeos: True
tabela diário (caso): soma de n_videos == vídeos sobre_o_caso: True
tabela pregão (caso): soma de n_videos == vídeos sobre_o_caso: True
tabela diário: n_outliers == vídeos com outlier_views: True
tabela pregão: n_outliers == vídeos com outlier_views: True
idade_dias negativa: 0
               min         max
fuso                          
-04:00  2026-10-18  2026-11-01
-05:00  2026-11-01  2026-11-18
data confere com hora_utc convertida: True
data_pregao sempre em dia útil: True
data_pregao nunca antes de data: True
idioma
en    8384
Name: count, dtype: int64
tema_origem
titulo_pessoa     6305
titulo_empresa    1230
descricao_tags     849
Name: count, dtype: int64
            data  

## Estatísticas para o relatório
Os números da documentação abaixo saem desta célula. Rode-a antes de citá-los.

In [ ]:
no_teto = [(ini, fim) for ini, fim in janelas if len(cache.get(ini, [])) >= TETO]
blocos = [b for ini, fim in no_teto for b in subjanelas(ini, fim)]
blocos_no_teto = sum(len(cache_sub.get(i, [])) >= TETO for i, _ in blocos)
sem_curtidas, sem_comentarios = int(df["curtidas"].isna().sum()), int(df["comentarios"].isna().sum())

print(f"Janelas diárias: {len(janelas)} | que atingiram o teto de {TETO} vídeos: {len(no_teto)}")
print(f"Refino em blocos de {JANELA_SUB_HORAS}h: {len(blocos)} blocos buscados | ainda no teto: {blocos_no_teto}")
print("Funil:", funil)
print(f"Sem curtidas: {sem_curtidas} ({sem_curtidas / len(df):.1%}) | "
      f"sem comentários: {sem_comentarios} ({sem_comentarios / len(df):.1%})")
print("Origem do idioma:", df["idioma_fonte"].value_counts().to_dict())
print("Origem do tema:", df["tema_origem"].value_counts().to_dict())
print(f"Sobre o caso: {int(df['sobre_o_caso'].sum())} de {len(df)} vídeos")
print(f"Outliers de views: {int(df['outlier_views'].sum())} | idade mediana na coleta: {df['idade_dias'].median()} dias")
print("Deslocados para o pregão seguinte:", int((df["data_pregao"] != df["data"]).sum()),
      "| pregões com cobertura completa:", int(diario_pregao["cobertura_completa"].sum()), "de", len(diario_pregao))


Janelas diárias: 32 | que atingiram o teto de 150 vídeos: 23
Refino em blocos de 6h: 93 blocos buscados | ainda no teto: 48
Funil: {'ids_encontrados': 11433, 'videos_brutos': 11433, 'duplicados': 0, 'fora_do_periodo': 0, 'fora_do_tema': 1177, 'fora_do_idioma': 1872, 'idioma_indefinido_mantidos': 0, 'videos_finais': 8384}
Sem curtidas: 162 (1.9%) | sem comentários: 48 (0.6%)
Origem do idioma: {'metadata': 8384}
Origem do tema: {'titulo_pessoa': 6305, 'titulo_empresa': 1230, 'descricao_tags': 849}
Sobre o caso: 2067 de 8384 vídeos
Outliers de views: 3 | idade mediana na coleta: 10.1 dias
Deslocados para o pregão seguinte: 4262 | pregões com cobertura completa: 22 de 24


## Documentação da fonte e limitações (para o relatório)
**Fonte:** YouTube Data API v3 (https://developers.google.com/youtube/v3), acessada na data da coluna `data_coleta`. Período: 18/10 a 18/11/2026, datas no fuso de Nova York.

**Por que junto com as outras bases:** a repercussão no YouTube, a cobertura de notícias e as ações da NVDA têm a data como chave comum, o que permite comparar atenção do público, da imprensa e do mercado dia a dia.

**Limitações** (os números vêm da célula "Estatísticas para o relatório"):
1. A busca é limitada a `MAX_PAGINAS` páginas de 50 vídeos por janela (teto de 150 com 3 páginas; é uma escolha do notebook para controlar a cota, a API permitiria mais). Os dias que atingem o teto são rebuscados em blocos de `JANELA_SUB_HORAS` horas. Se nenhum bloco ficar no teto, `n_videos` representa o volume de vídeos que a busca encontra; se algum bloco ainda atingir o teto, nesse dia o dataset continua sendo uma amostra dos vídeos mais relevantes (a busca ordena por relevância e tende a favorecer vídeos de maior audiência) e `n_videos` é um mínimo. Nesses dias, prefira `views` e `taxa_engajamento`. A célula "Estatísticas para o relatório" informa quantos blocos ficaram no teto. Mesmo sem teto, a busca do YouTube não garante listar todos os vídeos que citam o termo.
2. O filtro de tema seleciona vídeos sobre Jensen Huang e a NVIDIA em geral, não só sobre o caso: títulos com "NVIDIA" incluem reviews de GPU e análises de ações, e vídeos sem o nome no título só entram se a descrição ou as tags citarem "Jensen Huang". Para isolar o caso, a coluna `sobre_o_caso` marca os vídeos cujo título cita o escândalo (regra por palavras-chave, `CASO`), e há tabelas agregadas só com eles (`youtube_diario_caso.csv`, `youtube_diario_pregao_caso.csv`). A regra olha só o título, então pode deixar de marcar vídeos sobre o caso com títulos neutros e marcar algum fora do tema.
3. Curtidas ocultas e comentários desativados ficam como NaN nas colunas originais e valem 0 só no cálculo da taxa de engajamento, o que a subestima levemente.
4. As views são cumulativas até a data da coleta, então vídeos mais antigos tiveram mais tempo para acumular. `idade_dias` (por vídeo) e `idade_media_dias` (por dia) mostram esse efeito.
5. O idioma vem do que o canal declarou (`defaultAudioLanguage` ou `defaultLanguage`), que pode estar errado ou ausente; nesses casos usa-se detecção automática sobre título e descrição, menos confiável em textos curtos. Vídeos de idioma indefinido são mantidos (`idioma_fonte = indefinido`).
6. Publicações de fim de semana e depois das 16h (NY) são atribuídas ao próximo pregão, então a atenção de um dia de bolsa reúne o que foi publicado desde o fechamento anterior. Os pregões das pontas (`cobertura_completa` falso) têm cobertura parcial. O relatório deve informar quantos vídeos foram deslocados para o pregão seguinte.
7. Os vídeos virais (`outlier_views`) tendem a se concentrar nos dias do caso. Por isso `n_outliers` e `views_sem_outliers` servem como teste de robustez e não como resultado principal: tirar os outliers tira justamente o efeito que se quer medir.
